In [1]:
import sys
sys.path.append("..")

from src.data_loader import preparar_datos


In [2]:
# Cargar los datos preparados
X_train, X_test, y_train, y_test, preprocesador = preparar_datos(
    "../data/raw/dataset_practica_final.csv"
)

In [3]:
from sklearn.model_selection import train_test_split

# Separar ajuste y validacion
X_ajuste, X_val, y_ajuste, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train,
)

print("Entrenamiento:", X_ajuste.shape)
print("Validación:", X_val.shape)
print("Test reservado para C:", X_test.shape)

Entrenamiento: (76409, 29)
Validación: (19103, 29)
Test reservado para C: (23878, 29)


In [4]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Metricas de validación para la tabla comparativa
def crear_fila_comparativa(nombre, y_real, predicciones, probabilidades):
    return {
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_real, predicciones),
        "Precision": precision_score(y_real, predicciones, zero_division=0),
        "Recall": recall_score(y_real, predicciones, zero_division=0),
        "F1": f1_score(y_real, predicciones, zero_division=0),
        "ROC-AUC": roc_auc_score(y_real, probabilidades),
    }

In [5]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Regresión logística: crear el Pipeline y entrenar
# Cada modelo tiene su propia copia del preprocesador
modelo_logistico = Pipeline([
    ("preprocesado", clone(preprocesador)),
    ("modelo", LogisticRegression(max_iter=1000)),
])

modelo_logistico.fit(X_ajuste, y_ajuste)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesado', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](29,)","['hotel','lead_time','arrival_date_year',...,'total_of_special_requests', 'has_company','has_agent']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,29
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``r

In [6]:
# Regresión logística: obtener etiquetas, probabilidades y métricas en validación
predicciones_logistico = modelo_logistico.predict(X_val)
probabilidades_logistico = modelo_logistico.predict_proba(X_val)[:, 1]

resultado_logistico = crear_fila_comparativa(
    "Regresión logística", y_val, predicciones_logistico, probabilidades_logistico
)
pd.Series(resultado_logistico)

Modelo       Regresión logística
Accuracy                0.816102
Precision               0.809449
Recall                  0.658564
F1                      0.726253
ROC-AUC                 0.894776
dtype: object

In [7]:
from sklearn.tree import DecisionTreeClassifier

# Árbol de decisión: crear el Pipeline y entrenar
modelo_arbol = Pipeline([
    ("preprocesado", clone(preprocesador)),
    ("modelo", DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
    )),
])

modelo_arbol.fit(X_ajuste, y_ajuste)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesado', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](29,)","['hotel','lead_time','arrival_date_year',...,'total_of_special_requests', 'has_company','has_agent']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,29
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``r

In [8]:
# Árbol de decisión: obtener etiquetas, probabilidades y métricas en validación
predicciones_arbol = modelo_arbol.predict(X_val)
probabilidades_arbol = modelo_arbol.predict_proba(X_val)[:, 1]

resultado_arbol = crear_fila_comparativa(
    "Árbol de decisión", y_val, predicciones_arbol, probabilidades_arbol
)
pd.Series(resultado_arbol)

Modelo       Árbol de decisión
Accuracy              0.825158
Precision             0.789254
Recall                0.720322
F1                    0.753214
ROC-AUC               0.907949
dtype: object

In [9]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest: crear el Pipeline y entrenar
modelo_forest = Pipeline([
    ("preprocesado", clone(preprocesador)),
    ("modelo", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1,
    )),
])

modelo_forest.fit(X_ajuste, y_ajuste)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesado', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](29,)","['hotel','lead_time','arrival_date_year',...,'total_of_special_requests', 'has_company','has_agent']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,29
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``r

In [10]:
# Random Forest: obtener etiquetas, probabilidades y métricas en validación
predicciones_forest = modelo_forest.predict(X_val)
probabilidades_forest = modelo_forest.predict_proba(X_val)[:, 1]

resultado_forest = crear_fila_comparativa(
    "Random Forest", y_val, predicciones_forest, probabilidades_forest
)
pd.Series(resultado_forest)

Modelo       Random Forest
Accuracy          0.843794
Precision         0.885747
Recall            0.663934
F1                0.758966
ROC-AUC           0.928707
dtype: object

In [11]:
from xgboost import XGBClassifier

# XGBoost: crear el Pipeline y entrenar
modelo_xgboost = Pipeline([
    ("preprocesado", clone(preprocesador)),
    ("modelo", XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=4,
    )),
])

modelo_xgboost.fit(X_ajuste, y_ajuste)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesado', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](29,)","['hotel','lead_time','arrival_date_year',...,'total_of_special_requests', 'has_company','has_agent']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,29
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``r

In [12]:
# XGBoost: obtener etiquetas, probabilidades y métricas en validación
predicciones_xgboost = modelo_xgboost.predict(X_val)
probabilidades_xgboost = modelo_xgboost.predict_proba(X_val)[:, 1]

resultado_xgboost = crear_fila_comparativa(
    "XGBoost", y_val, predicciones_xgboost, probabilidades_xgboost
)
pd.Series(resultado_xgboost)

Modelo        XGBoost
Accuracy      0.86332
Precision    0.843198
Recall       0.775155
F1           0.807746
ROC-AUC      0.940812
dtype: object

In [13]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Keras: ajustar el preprocesado con entrenamiento y transformar validación
preprocesador_red = clone(preprocesador)
X_ajuste_red = preprocesador_red.fit_transform(X_ajuste)
X_val_red = preprocesador_red.transform(X_val)

# Convertir matrices dispersas a arrays densos
if hasattr(X_ajuste_red, "toarray"):
    X_ajuste_red = X_ajuste_red.toarray()
if hasattr(X_val_red, "toarray"):
    X_val_red = X_val_red.toarray()

# Aplicar un escalado adicional y convertir los datos a float32
escalador_red = StandardScaler()
X_ajuste_red = escalador_red.fit_transform(X_ajuste_red)
X_val_red = escalador_red.transform(X_val_red)
X_ajuste_red = X_ajuste_red.astype(np.float32)
X_val_red = X_val_red.astype(np.float32)

print("Entrenamiento:", X_ajuste_red.shape)
print("Validación:", X_val_red.shape)

Entrenamiento: (76409, 90)
Validación: (19103, 90)


In [14]:
import tensorflow as tf

# Keras: red con dos capas ocultas y salida de probabilidad
tf.keras.utils.set_random_seed(42)

modelo_red = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_ajuste_red.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

# Configurar optimizador y pérdida para clasificación binaria
modelo_red.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [15]:
# Keras: parar si val_loss no mejora durante 3 épocas y recuperar los mejores pesos
parada_temprana = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

# Entrenar hasta 20 épocas en lotes de 256 reservas
historial_red = modelo_red.fit(
    X_ajuste_red,
    y_ajuste,
    validation_data=(X_val_red, y_val),
    epochs=20,
    batch_size=256,
    callbacks=[parada_temprana],
    verbose=2,
)

Epoch 1/20
299/299 - 1s - 5ms/step - accuracy: 0.8031 - loss: 0.4011 - val_accuracy: 0.8284 - val_loss: 0.3534
Epoch 2/20
299/299 - 0s - 1ms/step - accuracy: 0.8386 - loss: 0.3364 - val_accuracy: 0.8404 - val_loss: 0.3334
Epoch 3/20
299/299 - 0s - 1ms/step - accuracy: 0.8494 - loss: 0.3191 - val_accuracy: 0.8447 - val_loss: 0.3230
Epoch 4/20
299/299 - 0s - 1ms/step - accuracy: 0.8552 - loss: 0.3087 - val_accuracy: 0.8480 - val_loss: 0.3169
Epoch 5/20
299/299 - 0s - 1ms/step - accuracy: 0.8590 - loss: 0.3016 - val_accuracy: 0.8495 - val_loss: 0.3132
Epoch 6/20
299/299 - 0s - 1ms/step - accuracy: 0.8622 - loss: 0.2964 - val_accuracy: 0.8508 - val_loss: 0.3102
Epoch 7/20
299/299 - 0s - 1ms/step - accuracy: 0.8637 - loss: 0.2921 - val_accuracy: 0.8520 - val_loss: 0.3084
Epoch 8/20
299/299 - 0s - 1ms/step - accuracy: 0.8654 - loss: 0.2886 - val_accuracy: 0.8541 - val_loss: 0.3072
Epoch 9/20
299/299 - 0s - 1ms/step - accuracy: 0.8668 - loss: 0.2854 - val_accuracy: 0.8548 - val_loss: 0.3062
E

In [16]:
# Keras: convertir probabilidades en etiquetas con umbral 0,5
probabilidades_red = modelo_red.predict(X_val_red, verbose=0).ravel()
predicciones_red = (probabilidades_red >= 0.5).astype(int)

# Registrar las mismas métricas que para los modelos anteriores
resultado_red = crear_fila_comparativa(
    "Red Keras", y_val, predicciones_red, probabilidades_red
)
pd.Series(resultado_red)

Modelo       Red Keras
Accuracy       0.85997
Precision     0.819144
Recall        0.798191
F1            0.808532
ROC-AUC       0.936396
dtype: object

In [17]:
# Tabla comparativa de los cinco modelos sobre validación
resultados = [
    resultado_logistico,
    resultado_arbol,
    resultado_forest,
    resultado_xgboost,
    resultado_red,
]
tabla_comparativa = pd.DataFrame(resultados).set_index("Modelo")
tabla_comparativa.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC
Modelo,,,,,
Regresión logística,0.816,0.809,0.659,0.726,0.895
Árbol de decisión,0.825,0.789,0.720,0.753,0.908
Random Forest,0.844,0.886,0.664,0.759,0.929
XGBoost,0.863,0.843,0.775,0.808,0.941
Red Keras,0.860,0.819,0.798,0.809,0.936


In [18]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# GridSearchCV: buscar por separado los parámetros de cada modelo
# Usar solo ajuste, con 3 particiones y F1 como criterio
cv_grid = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [19]:
# Regresión logística: probar distintas fuerzas de regularización
grid_logistico = GridSearchCV(
    estimator=clone(modelo_logistico),
    param_grid={"modelo__C": [0.1, 1.0, 10.0]},
    scoring="f1",
    cv=cv_grid,
    n_jobs=2,
    error_score="raise",
)
grid_logistico.fit(X_ajuste, y_ajuste)
print("Parámetros elegidos:", grid_logistico.best_params_)
print(f"F1 medio de validación cruzada: {grid_logistico.best_score_:.3f}")

Parámetros elegidos: {'modelo__C': 1.0}
F1 medio de validación cruzada: 0.724


In [20]:
# Árbol: probar profundidad y mínimo de muestras por hoja
grid_arbol = GridSearchCV(
    estimator=clone(modelo_arbol),
    param_grid={
        "modelo__max_depth": [8, 12],
        "modelo__min_samples_leaf": [10, 20],
    },
    scoring="f1",
    cv=cv_grid,
    n_jobs=2,
    error_score="raise",
)
grid_arbol.fit(X_ajuste, y_ajuste)
print("Parámetros elegidos:", grid_arbol.best_params_)
print(f"F1 medio de validación cruzada: {grid_arbol.best_score_:.3f}")

Parámetros elegidos: {'modelo__max_depth': 12, 'modelo__min_samples_leaf': 10}
F1 medio de validación cruzada: 0.777


In [21]:
# Random Forest: probar profundidad y mínimo de muestras por hoja
# Cada bosque usa un proceso; GridSearchCV ejecuta dos ajustes en paralelo
grid_forest = GridSearchCV(
    estimator=clone(modelo_forest).set_params(modelo__n_jobs=1),
    param_grid={
        "modelo__max_depth": [8, 12],
        "modelo__min_samples_leaf": [5, 10],
    },
    scoring="f1",
    cv=cv_grid,
    n_jobs=2,
    error_score="raise",
)
grid_forest.fit(X_ajuste, y_ajuste)
print("Parámetros elegidos:", grid_forest.best_params_)
print(f"F1 medio de validación cruzada: {grid_forest.best_score_:.3f}")

Parámetros elegidos: {'modelo__max_depth': 12, 'modelo__min_samples_leaf': 5}
F1 medio de validación cruzada: 0.762


In [22]:
# XGBoost: probar profundidad y tasa de aprendizaje
grid_xgboost = GridSearchCV(
    estimator=clone(modelo_xgboost),
    param_grid={
        "modelo__max_depth": [3, 5],
        "modelo__learning_rate": [0.05, 0.1],
    },
    scoring="f1",
    cv=cv_grid,
    n_jobs=1,
    error_score="raise",
)
grid_xgboost.fit(X_ajuste, y_ajuste)
print("Parámetros elegidos:", grid_xgboost.best_params_)
print(f"F1 medio de validación cruzada: {grid_xgboost.best_score_:.3f}")

Parámetros elegidos: {'modelo__learning_rate': 0.1, 'modelo__max_depth': 5}
F1 medio de validación cruzada: 0.809


In [23]:
from src.model_trainer import ClasificadorKeras

# Keras: el adaptador ajusta el preprocesado dentro de cada partición
# Probar épocas y tamaño del lote; entrenar las redes de una en una
# GridSearchCV utiliza sus propias particiones de ajuste y validación interna
grid_red = GridSearchCV(
    estimator=ClasificadorKeras(preprocesador),
    param_grid={"epochs": [10, 20], "batch_size": [128, 256]},
    scoring="f1",
    cv=cv_grid,
    n_jobs=1,
    error_score="raise",
)
grid_red.fit(X_ajuste, y_ajuste)
print("Parámetros elegidos:", grid_red.best_params_)
print(f"F1 medio de validación cruzada: {grid_red.best_score_:.3f}")

Parámetros elegidos: {'batch_size': 128, 'epochs': 20}
F1 medio de validación cruzada: 0.796


In [24]:
# Comparar los cinco modelos ajustados con los cinco modelos base
# best_estimator_ ya está entrenado con la combinación elegida por la búsqueda
modelos_grid = {
    "Regresión logística (GridSearchCV)": grid_logistico.best_estimator_,
    "Árbol de decisión (GridSearchCV)": grid_arbol.best_estimator_,
    "Random Forest (GridSearchCV)": grid_forest.best_estimator_,
    "XGBoost (GridSearchCV)": grid_xgboost.best_estimator_,
    "Red Keras (GridSearchCV)": grid_red.best_estimator_,
}

resultados_grid = []
for nombre, modelo in modelos_grid.items():
    predicciones = modelo.predict(X_val)
    probabilidades = modelo.predict_proba(X_val)[:, 1]
    resultados_grid.append(
        crear_fila_comparativa(nombre, y_val, predicciones, probabilidades)
    )

tabla_grid = pd.DataFrame(resultados_grid).set_index("Modelo")
tabla_completa = pd.concat([tabla_comparativa, tabla_grid])
tabla_completa.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC
Modelo,,,,,
Regresión logística,0.816,0.809,0.659,0.726,0.895
Árbol de decisión,0.825,0.789,0.720,0.753,0.908
Random Forest,0.844,0.886,0.664,0.759,0.929
XGBoost,0.863,0.843,0.775,0.808,0.941
Red Keras,0.860,0.819,0.798,0.809,0.936
Regresión logística (GridSearchCV),0.816,0.809,0.659,0.726,0.895
Árbol de decisión (GridSearchCV),0.846,0.821,0.747,0.783,0.921
Random Forest (GridSearchCV),0.846,0.889,0.666,0.762,0.930
XGBoost (GridSearchCV),0.863,0.843,0.775,0.808,0.941


In [25]:
# Comparar los parámetros iniciales con los elegidos por GridSearchCV
parametros_base = [
    {"C": modelo_logistico.named_steps["modelo"].C},
    {"max_depth": modelo_arbol.named_steps["modelo"].max_depth,
     "min_samples_leaf": modelo_arbol.named_steps["modelo"].min_samples_leaf},
    {"max_depth": modelo_forest.named_steps["modelo"].max_depth,
     "min_samples_leaf": modelo_forest.named_steps["modelo"].min_samples_leaf},
    {"max_depth": modelo_xgboost.named_steps["modelo"].max_depth,
     "learning_rate": modelo_xgboost.named_steps["modelo"].learning_rate},
    {"epochs": 20, "batch_size": 256},
]
busquedas = [grid_logistico, grid_arbol, grid_forest, grid_xgboost, grid_red]

comparacion_parametros = []
for nombre, iniciales, busqueda in zip(tabla_comparativa.index, parametros_base, busquedas):
    elegidos = {p.replace("modelo__", ""): v for p, v in busqueda.best_params_.items()}
    comparacion_parametros.append({"Modelo": nombre, "Iniciales": iniciales, "GridSearchCV": elegidos})

tabla_parametros = pd.DataFrame(comparacion_parametros).set_index("Modelo")
tabla_parametros

,Iniciales,GridSearchCV
Modelo,,
Regresión logística,{'C': 1.0},{'C': 1.0}
Árbol de decisión,"{'max_depth': 8, 'min_samples_leaf': 20}","{'max_depth': 12, 'min_samples_leaf': 10}"
Random Forest,"{'max_depth': 12, 'min_samples_leaf': 10}","{'max_depth': 12, 'min_samples_leaf': 5}"
XGBoost,"{'max_depth': 5, 'learning_rate': 0.1}","{'learning_rate': 0.1, 'max_depth': 5}"
Red Keras,"{'epochs': 20, 'batch_size': 256}","{'batch_size': 128, 'epochs': 20}"


In [26]:
# Cambios en validación: positivo = mejora; negativo = empeora
tabla_grid_alineada = tabla_grid.rename(
    index=lambda nombre: nombre.replace(" (GridSearchCV)", "")
)
tabla_mejoras = tabla_grid_alineada - tabla_comparativa
tabla_mejoras.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC
Modelo,,,,,
Regresión logística,0.000,0.000,0.000,0.000,0.000
Árbol de decisión,0.021,0.032,0.027,0.029,0.013
Random Forest,0.002,0.003,0.002,0.003,0.001
XGBoost,0.000,0.000,0.000,0.000,0.000
Red Keras,0.002,0.004,0.001,0.003,0.001
